In [1]:
import os
from PIL import Image
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

import albumentations as A
from albumentations.pytorch import ToTensorV2

images_path = "/mnt/c/Users/lloyd/OneDrive/Documents/UP/8vo_Semestre/ML_II/corsican/images"
masks_path = "/mnt/c/Users/lloyd/OneDrive/Documents/UP/8vo_Semestre/ML_II/corsican/masks"

# path to save models, history and results
SAVE_DIR = "/mnt/c/Users/lloyd/OneDrive/Documents/UP/8vo_Semestre/ML_II/corsican/corsican_model"
os.makedirs(SAVE_DIR, exist_ok=True)


In [2]:
class FireDataset(Dataset):
    def __init__(self, images_path, masks_path, transform=None):
        self.images_path = images_path
        self.masks_path = masks_path
        self.transform = transform

        self.images = sorted(os.listdir(images_path))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
      img_name = self.images[idx]

      img_path = os.path.join(self.images_path, img_name)
      mask_name = img_name.replace("_rgb", "_gt")
      mask_path = os.path.join(self.masks_path, mask_name)

      image = np.array(Image.open(img_path).convert("RGB"))
      mask = np.array(Image.open(mask_path).convert("L"))

      mask = (mask > 0).astype("float32")

      if self.transform:
          augmented = self.transform(image=image, mask=mask)
          image = augmented["image"]
          mask = augmented["mask"].unsqueeze(0)

      if image.shape[1] != 256 or image.shape[2] != 256:
          image = torch.nn.functional.interpolate(
              image.unsqueeze(0), size=(256, 256), mode="bilinear", align_corners=False
          ).squeeze(0)

      if mask.shape[1] != 256 or mask.shape[2] != 256:
          mask = torch.nn.functional.interpolate(
              mask.unsqueeze(0), size=(256, 256), mode="nearest"
          ).squeeze(0)

      return image, mask

In [3]:
#Data augmenattion
train_transform  = A.Compose([
    A.HorizontalFlip(p=0.5),            # random flip
    A.VerticalFlip(p=0.5),
    A.Resize(256, 256),                 # resize
    A.HueSaturationValue(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

test_transform = val_transform

In [4]:
!pip install segmentation-models-pytorch -q


In [5]:
#loss function
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, preds, targets):
        preds = torch.sigmoid(preds)

        preds = preds.view(-1)
        targets = targets.view(-1)

        intersection = (preds * targets).sum()
        dice = (2. * intersection + self.smooth) / (
            preds.sum() + targets.sum() + self.smooth
        )

        return 1 - dice

In [6]:
from torch.utils.data import Subset
import random

def create_data_splits(images_path, masks_path, train_transform, val_transform, data_percentage=1.0):
    """
    Create train/val/test splits using a percentage of the full dataset.
    Args:
        data_percentage: 0.25, 0.50 or 1.0
    Returns:
        train_dataset, val_dataset, test_dataset
    """
    total_size = len(FireDataset(images_path, masks_path))
    used_size  = int(total_size * data_percentage)

    train_size = int(0.70 * used_size)
    val_size   = int(0.15 * used_size)
    test_size  = used_size - train_size - val_size

    print(f'\nUsing {data_percentage*100:.0f}% of dataset:')
    print(f'  Total images : {total_size}')
    print(f'  Used images  : {used_size}')
    print(f'  Train        : {train_size}')
    print(f'  Val          : {val_size}')
    print(f'  Test         : {test_size}')

    full_train = FireDataset(images_path, masks_path, transform=train_transform)
    full_val   = FireDataset(images_path, masks_path, transform=val_transform)
    full_test  = FireDataset(images_path, masks_path, transform=val_transform)

    indices = list(range(total_size))
    random.seed(42)
    random.shuffle(indices)
    indices = indices[:used_size]

    train_idx = indices[:train_size]
    val_idx   = indices[train_size:train_size + val_size]
    test_idx  = indices[train_size + val_size:]

    train_dataset = Subset(full_train, train_idx)
    val_dataset   = Subset(full_val,   val_idx)
    test_dataset  = Subset(full_test,  test_idx)

    return train_dataset, val_dataset, test_dataset


In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

torch.manual_seed(42)
import numpy as np
np.random.seed(42)
random.seed(42)

Using device: cuda


In [8]:
def get_metrics(preds, targets, threshold=0.5):
    preds = torch.sigmoid(preds)
    preds = (preds > threshold).float()

    preds = preds.view(-1)
    targets = targets.view(-1)

    TP = (preds * targets).sum()
    TN = ((1 - preds) * (1 - targets)).sum()
    FP = (preds * (1 - targets)).sum()
    FN = ((1 - preds) * targets).sum()

    # F1 Score
    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)
    f1 = (2 * precision * recall) / (precision + recall + 1e-8)

    # mIoU
    miou = TP / (TP + FP + FN + 1e-8)

    # MCC
    mcc = (TP * TN - FP * FN) / torch.sqrt(
        (TP + FP) * (TP + FN) * (TN + FP) * (TN + FN) + 1e-8
    )

    # HAF (igual a IoU en binario, pero lo dejamos explícito)
    haf = TP / (TP + FP + FN + 1e-8)

    return f1.item(), miou.item(), mcc.item(), haf.item()

In [9]:
class EarlyStopping:
    """
    Stops training when val_loss does not improve for `patience` consecutive epochs.
    Also saves the best model weights.
    """
    def __init__(self, patience=10, min_delta=1e-4, verbose=True):
        """
        Args:
            patience  : epochs to wait without improvement before stopping
            min_delta : minimum change to count as an improvement
            verbose   : print a message each time the counter increases
        """
        self.patience   = patience
        self.min_delta  = min_delta
        self.verbose    = verbose
        self.counter    = 0
        self.best_loss  = None
        self.best_weights = None
        self.stop       = False

    def __call__(self, val_loss, model):
        if self.best_loss is None or val_loss < self.best_loss - self.min_delta:
            # Improvement found
            self.best_loss    = val_loss
            self.best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.counter      = 0
        else:
            self.counter += 1
            if self.verbose:
                print(f"  EarlyStopping: no improvement for {self.counter}/{self.patience} epochs "
                      f"(best val_loss={self.best_loss:.4f})")
            if self.counter >= self.patience:
                self.stop = True

    def restore_best_weights(self, model):
        """Load the best weights back into the model."""
        model.load_state_dict(self.best_weights)
        print(f"  Best weights restored (val_loss={self.best_loss:.4f})")

In [10]:
import segmentation_models_pytorch as smp
import json as _json

BATCH_SIZE = 8
NUM_EPOCHS = 50

def train_model(train_loader, val_loader, test_loader, data_percentage, num_epochs=50):
    pct_tag      = int(data_percentage * 100)
    pct_str      = f'{pct_tag}%'
    model_path   = os.path.join(SAVE_DIR, f'deeplabv3plus_{pct_tag}.pth')
    history_path = os.path.join(SAVE_DIR, f'deeplabv3plus_history_{pct_tag}.json')
    results_path = os.path.join(SAVE_DIR, f'deeplabv3plus_results_{pct_tag}.json')

    # ── Si ya existe todo, cargar y saltar entrenamiento ──
    if os.path.exists(model_path) and os.path.exists(history_path) and os.path.exists(results_path):
        print(f'\n{"="*70}')
        print(f'Modelo {pct_str} ya entrenado. Cargando desde Drive...')
        print(f'{"="*70}\n')

        model = smp.DeepLabV3Plus(
            encoder_name='resnet50',
            encoder_weights='imagenet',
            in_channels=3,
            classes=1,
            activation=None
        ).to(device)

        checkpoint = torch.load(model_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()

        with open(history_path) as f:
            history = _json.load(f)
        with open(results_path) as f:
            test_results = _json.load(f)

        print(f"  Val mIoU guardado : {checkpoint['val_miou']:.4f}")
        print(f"  Test mIoU guardado: {test_results['test_miou']:.4f}")
        return history, test_results, model

    # ── Entrenamiento ──
    print(f'\n{"="*70}')
    print(f'TRAINING DeepLabV3+ (ResNet50) WITH {pct_str} OF DATASET')
    print(f'{"="*70}\n')

    model = smp.DeepLabV3Plus(
        encoder_name='resnet50',
        encoder_weights='imagenet',
        in_channels=3,
        classes=1,
        activation=None
    ).to(device)

    criterion      = DiceLoss()
    optimizer      = torch.optim.Adam(model.parameters(), lr=1e-4)
    early_stopping = EarlyStopping(patience=10, min_delta=1e-4, verbose=True)

    best_val_miou = 0.0
    history = {'train_loss':[], 'val_loss':[], 'train_f1':[], 'val_f1':[],
               'train_miou':[], 'val_miou':[], 'train_mcc':[], 'val_mcc':[]}

    for epoch in range(num_epochs):
        # --- TRAIN ---
        model.train()
        train_loss, train_metrics = 0, [0]*4
        for images, masks in train_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            metrics = get_metrics(outputs, masks)
            train_metrics = [x + y for x, y in zip(train_metrics, metrics)]
        train_loss /= len(train_loader)
        train_metrics = [x / len(train_loader) for x in train_metrics]

        # --- VALIDATION ---
        model.eval()
        val_loss, val_metrics = 0, [0]*4
        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.to(device), masks.to(device)
                outputs = model(images)
                loss = criterion(outputs, masks)
                val_loss += loss.item()
                metrics = get_metrics(outputs, masks)
                val_metrics = [x + y for x, y in zip(val_metrics, metrics)]
        val_loss /= len(val_loader)
        val_metrics = [x / len(val_loader) for x in val_metrics]

        # --- LOG ---
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_f1'].append(train_metrics[0])
        history['val_f1'].append(val_metrics[0])
        history['train_miou'].append(train_metrics[1])
        history['val_miou'].append(val_metrics[1])
        history['train_mcc'].append(train_metrics[2])
        history['val_mcc'].append(val_metrics[2])

        # --- GUARDAR MEJOR MODELO ---
        val_miou = val_metrics[1]
        if val_miou > best_val_miou:
            best_val_miou = val_miou
            torch.save({'epoch': epoch,
                        'model_state_dict': model.state_dict(),
                        'val_miou': val_miou}, model_path)
            print(f'  [Epoch {epoch+1}] Nuevo mejor modelo guardado (val_mIoU={val_miou:.4f})')

        # --- PRINT cada 5 epochs ---
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}]')
            print(f'  Train - Loss: {train_loss:.4f}, F1: {train_metrics[0]:.4f}, mIoU: {train_metrics[1]:.4f}')
            print(f'  Val   - Loss: {val_loss:.4f}, F1: {val_metrics[0]:.4f}, mIoU: {val_metrics[1]:.4f}')

        # --- EARLY STOPPING ---
        early_stopping(val_loss, model)
        if early_stopping.stop:
            print(f'\n  Early stopping at epoch {epoch+1}')
            break

    early_stopping.restore_best_weights(model)

    # Recargar desde archivo para consistencia
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    # --- TEST ---
    test_loss, test_metrics = 0, [0]*4
    with torch.no_grad():
        for images, masks in test_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            test_loss += loss.item()
            metrics = get_metrics(outputs, masks)
            test_metrics = [x + y for x, y in zip(test_metrics, metrics)]
    test_loss /= len(test_loader)
    test_metrics = [x / len(test_loader) for x in test_metrics]

    names = ['F1', 'mIoU', 'MCC', 'HAF']
    print(f'\n{"="*70}')
    print(f'RESULTS - {pct_str} Dataset (DeepLabV3+ ResNet50)')
    print(f'{"="*70}')
    print(f"\n{'Metric':<12} {'Train':>10} {'Val':>10} {'Test':>10}")
    print('-' * 45)
    for i, name in enumerate(names):
        print(f'{name:<12} {train_metrics[i]:>10.4f} {val_metrics[i]:>10.4f} {test_metrics[i]:>10.4f}')
    print('-' * 45)
    print(f"{'Loss':<12} {train_loss:>10.4f} {val_loss:>10.4f} {test_loss:>10.4f}")

    test_results = {
        'test_loss': test_loss,
        'test_f1':   test_metrics[0],
        'test_miou': test_metrics[1],
        'test_mcc':  test_metrics[2],
        'test_haf':  test_metrics[3],
    }

    # Guardar historial y resultados
    with open(history_path, 'w') as f:
        _json.dump(history, f)
    with open(results_path, 'w') as f:
        _json.dump(test_results, f)

    print(f'\n  Guardado en Drive:')
    print(f'    Modelo    : deeplabv3plus_{pct_tag}.pth')
    print(f'    Historial : deeplabv3plus_history_{pct_tag}.json')
    print(f'    Resultados: deeplabv3plus_results_{pct_tag}.json')

    return history, test_results, model


In [11]:
train_25, val_25, test_25 = create_data_splits(
    images_path, masks_path, train_transform, val_transform, data_percentage=0.25)

train_loader_25 = DataLoader(train_25, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader_25   = DataLoader(val_25,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader_25  = DataLoader(test_25,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

history_25, test_results_25, model_25 = train_model(
    train_loader_25, val_loader_25, test_loader_25,
    data_percentage=0.25, num_epochs=NUM_EPOCHS)



Using 25% of dataset:
  Total images : 500
  Used images  : 125
  Train        : 87
  Val          : 18
  Test         : 20

Modelo 25% ya entrenado. Cargando desde Drive...

  Val mIoU guardado : 0.8634
  Test mIoU guardado: 0.8550


In [ ]:
train_50, val_50, test_50 = create_data_splits(
    images_path, masks_path, train_transform, val_transform, data_percentage=0.50)

train_loader_50 = DataLoader(train_50, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader_50   = DataLoader(val_50,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader_50  = DataLoader(test_50,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

history_50, test_results_50, model_50 = train_model(
    train_loader_50, val_loader_50, test_loader_50,
    data_percentage=0.50, num_epochs=NUM_EPOCHS)



Using 50% of dataset:
  Total images : 500
  Used images  : 250
  Train        : 175
  Val          : 37
  Test         : 38

TRAINING DeepLabV3+ (ResNet50) WITH 50% OF DATASET



In [ ]:
train_100, val_100, test_100 = create_data_splits(
    images_path, masks_path, train_transform, val_transform, data_percentage=1.0)

train_loader_100 = DataLoader(train_100, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader_100   = DataLoader(val_100,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader_100  = DataLoader(test_100,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

history_100, test_results_100, model_100 = train_model(
    train_loader_100, val_loader_100, test_loader_100,
    data_percentage=1.0, num_epochs=NUM_EPOCHS)


In [ ]:
all_test_results = {'25%': test_results_25, '50%': test_results_50, '100%': test_results_100}

print('=' * 80)
print('SUMMARY: TEST SET RESULTS FOR DIFFERENT DATASET SIZES')
print('=' * 80)
print(f"{'Dataset':<12} {'Loss':<10} {'F1':<10} {'mIoU':<10} {'MCC':<10} {'HAF':<10}")
print('-' * 80)
for pct in ['25%', '50%', '100%']:
    r = all_test_results[pct]
    print(f"{pct:<12} {r['test_loss']:<10.4f} {r['test_f1']:<10.4f} "
          f"{r['test_miou']:<10.4f} {r['test_mcc']:<10.4f} {r['test_haf']:<10.4f}")
print('=' * 80)


In [ ]:
import matplotlib.pyplot as plt

all_histories = {'25%': history_25, '50%': history_50, '100%': history_100}
titles = ['Loss', 'F1 Score', 'mIoU', 'MCC']
keys   = [('train_loss','val_loss'), ('train_f1','val_f1'),
          ('train_miou','val_miou'), ('train_mcc','val_mcc')]
colors = {'25%':'blue', '50%':'orange', '100%':'green'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, title, (tk, vk) in zip(axes, titles, keys):
    for pct, hist in all_histories.items():
        c = colors[pct]
        ax.plot(hist[tk], color=c, linestyle='-',  label=f'{pct} train')
        ax.plot(hist[vk], color=c, linestyle='--', label=f'{pct} val')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=7)
    ax.grid(True)

plt.suptitle('DeepLabV3+ + ResNet50 — Training curves (25% / 50% / 100%)', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import random

model_100.eval()

# Samplear índices aleatorios del test set (diferente cada vez)
num_imgs = 8
test_dataset = test_loader_100.dataset
random_indices = random.sample(range(len(test_dataset)), num_imgs)

# Construir batch manualmente
images_list, masks_list = [], []
for idx in random_indices:
    img, mask = test_dataset[idx]
    images_list.append(img)
    masks_list.append(mask)

images = torch.stack(images_list).to(device)
masks  = torch.stack(masks_list)

with torch.no_grad():
    outputs = model_100(images)
    preds = torch.sigmoid(outputs)
    preds = (preds > 0.5).float()

# Desnormalizar para visualizar
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

fig, axes = plt.subplots(num_imgs, 3, figsize=(12, num_imgs * 4))

for i in range(num_imgs):
    img       = images[i].cpu() * std + mean
    img       = img.permute(1, 2, 0).clamp(0, 1)
    mask_real = masks[i].cpu().squeeze()
    mask_pred = preds[i].cpu().squeeze()

    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f"Original image (idx {random_indices[i]})")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(mask_real, cmap="gray")
    axes[i, 1].set_title("Real mask")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(mask_pred, cmap="gray")
    axes[i, 2].set_title("Predicted mask")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()